
# AI Credit Risk Prediction

 
Notebook 4 — Model Training

Author : Owaish Ansari

Purpose: ML_flow Tracking.



In [1]:
import os
import sys

project_root = r"C:\Users\Asus\AI_Credit_Risk_Prediction"

if project_root not in sys.path:
    sys.path.insert(0, project_root)

print("Project Root Added")

Project Root Added


In [2]:
import importlib.util

print(importlib.util.find_spec("src"))
print(importlib.util.find_spec("src.database"))
print(importlib.util.find_spec("src.database.postgres"))

ModuleSpec(name='src', loader=<_frozen_importlib_external.SourceFileLoader object at 0x000001D0B6E9EB10>, origin='C:\\Users\\Asus\\AI_Credit_Risk_Prediction\\src\\__init__.py', submodule_search_locations=['C:\\Users\\Asus\\AI_Credit_Risk_Prediction\\src'])
ModuleSpec(name='src.database', loader=<_frozen_importlib_external.SourceFileLoader object at 0x000001D0B6E9E4B0>, origin='C:\\Users\\Asus\\AI_Credit_Risk_Prediction\\src\\database\\__init__.py', submodule_search_locations=['C:\\Users\\Asus\\AI_Credit_Risk_Prediction\\src\\database'])
ModuleSpec(name='src.database.postgres', loader=<_frozen_importlib_external.SourceFileLoader object at 0x000001D0B6E9E4B0>, origin='C:\\Users\\Asus\\AI_Credit_Risk_Prediction\\src\\database\\postgres.py')


In [3]:
import mlflow
import mlflow.sklearn

import matplotlib.pyplot as plt

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    RocCurveDisplay
)

In [4]:
from src.database.postgres import load_table

final_dataset = load_table("final_dataset")

print(final_dataset.shape)

final_dataset Loaded
(307511, 244)


In [5]:
X = final_dataset.drop(columns=["TARGET"])

y = final_dataset["TARGET"]

print(X.shape)
print(y.shape)

(307511, 243)
(307511,)


In [8]:
from sklearn.model_selection import train_test_split


X_train, X_test, y_train, y_test = train_test_split(

    X,
    y,

    test_size=0.20,

    random_state=42,

    stratify=y

)

In [10]:
import joblib

model = joblib.load("../models/lightgbm_model.pkl")

preprocessor = joblib.load("../models/preprocessor.pkl")

threshold = joblib.load("../models/threshold.pkl")

In [11]:
X_train_final = preprocessor.transform(X_train)

X_test_final = preprocessor.transform(X_test)

In [12]:
y_prob = model.predict_proba(X_test_final)[:,1]

y_pred = (

    y_prob >= threshold

).astype(int)

In [13]:
accuracy = accuracy_score(y_test, y_pred)

precision = precision_score(y_test, y_pred)

recall = recall_score(y_test, y_pred)

f1 = f1_score(y_test, y_pred)

roc_auc = roc_auc_score(y_test, y_prob)

In [15]:
mlflow.set_experiment(
    "AI_Credit_Risk_Prediction"
)

with mlflow.start_run(
    run_name="LightGBM_Final_Model"
):
    print("MLflow Run Started")

MLflow Run Started


In [16]:
    mlflow.log_param("Model", "LightGBM")

    mlflow.log_param("Threshold", float(threshold))

    mlflow.log_param("Train Samples", len(X_train))

    mlflow.log_param("Test Samples", len(X_test))

    mlflow.log_param("Features", X.shape[1])

243

In [17]:
    mlflow.log_metric("Accuracy", accuracy)

    mlflow.log_metric("Precision", precision)

    mlflow.log_metric("Recall", recall)

    mlflow.log_metric("F1", f1)

    mlflow.log_metric("ROC_AUC", roc_auc)

In [18]:
    fig, ax = plt.subplots(figsize=(6,6))

    ConfusionMatrixDisplay(
        confusion_matrix(y_test, y_pred)
    ).plot(ax=ax)

    plt.savefig("confusion_matrix.png")

    plt.close()

    mlflow.log_artifact("confusion_matrix.png")

In [19]:
    fig, ax = plt.subplots(figsize=(6,6))

    RocCurveDisplay.from_predictions(
        y_test,
        y_prob,
        ax=ax
    )

    plt.savefig("roc_curve.png")

    plt.close()

    mlflow.log_artifact("roc_curve.png")

In [22]:
import mlflow.sklearn

mlflow.sklearn.log_model(
    sk_model=model,
    artifact_path="lightgbm_model",
    skops_trusted_types=[
        "collections.OrderedDict",
        "lightgbm.basic.Booster",
        "lightgbm.sklearn.LGBMClassifier"
    ]
)

2026/08/02 08:39:10 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


In [23]:
import mlflow
import lightgbm

print("MLflow:", mlflow.__version__)
print("LightGBM:", lightgbm.__version__)

MLflow: 3.15.0
LightGBM: 4.7.0


In [24]:
print("MLflow Run Completed Successfully")

MLflow Run Completed Successfully


In [32]:
import os

print(os.getcwd())

C:\Users\Asus\AI_Credit_Risk_Prediction\notebooks


In [33]:
print(os.path.abspath("templates"))

C:\Users\Asus\AI_Credit_Risk_Prediction\notebooks\templates


In [34]:
import os

print(os.listdir("../templates"))

['.ipynb_checkpoints', 'index.html', 'result.html']


In [35]:
import os

print(os.path.abspath("../templates"))
print(os.listdir("../templates"))

C:\Users\Asus\AI_Credit_Risk_Prediction\templates
['.ipynb_checkpoints', 'index.html', 'result.html']
